<a id="summary"></a>

# EDA — Validation Against Planted Segments (Synthetic Dataset Only)

> ⚠️ This notebook is only meaningful because the dataset is synthetic.
> `true_segment` labels are used exclusively here for generator validation.
> Notebooks 1 and 2 follow a discovery-first approach (no labels used).

**Purpose:** Confirm that the synthetic data generator (`faker_base_generation.py`) produced the correct segment proportions, demographic correlations, behavioral patterns, and retention curves. Every check in this notebook compares an *observed* statistic against an *expected* value derived from the generator's own parameters.

**Planted segments:** `high_value_active` (20%), `mid_value_regular` (30%), `low_value_dormant` (30%), `at_risk_churner` (20%) — total 8,000 customers.

---

## Jump to:
- [Part 1 — Population Balance](#part1)
- [Part 2 — Demographic Validators](#part2)
- [Part 3 — Behavioral Validators](#part3)
- [Part 4 — Retention Validators](#part4)
- [Part 5 — Generator Audit Summary](#part5)

---

## Setup — imports and data load

This notebook is self-contained. It loads `customers_raw` and `transactions_raw` directly from Supabase.

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import PercentFormatter
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [2]:
# override=True ensures .env changes are picked up without restarting the kernel
load_dotenv(override=True)

DATABASE_URL = os.environ["SUPABASE_DATABASE_URL"]

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,  # helps avoid stale connections in notebooks
)

sql = text("""
SELECT
  customer_id,
  name,
  email,
  age,
  state,
  registration_date,
  acquisition_channel,
  acquisition_cost,
  true_segment
FROM public.customers_raw
ORDER BY registration_date DESC
""")

df_customers = pd.read_sql(sql, engine)
df_customers.head()

,customer_id,name,email,age,state,registration_date,acquisition_channel,acquisition_cost,true_segment
0,06761548-609b-4cf9-a61f-5ac2fc1e8036,Maya Abreu,maya.abreu@yahoo.com.br,41,RJ,2026-02-28 00:00:00+00:00,organic,13.17,low_value_dormant
1,0d019d34-81ad-4cd5-8c47-d5e812d8f02c,Dr. João Vitor das Neves,dr.neves7665@outlook.com,18,BA,2026-02-27 00:00:00+00:00,organic,13.32,low_value_dormant
2,e12b4555-965d-4b55-b681-e84c9387ecba,Miguel Caldeira,miguel.caldeira@outlook.com,18,SP,2026-02-26 00:00:00+00:00,referral,58.68,mid_value_regular
3,c6544c38-2606-4c88-8633-7986d1dd007d,Theodoro Azevedo,theodoro.azevedo@gmail.com,26,RS,2026-02-26 00:00:00+00:00,organic,10.66,at_risk_churner
4,48f363fd-ce49-418e-b148-db68bc35edee,Enrico da Paz,enrico.paz@yahoo.com.br,46,SP,2026-02-25 00:00:00+00:00,paid_ads,202.79,high_value_active


In [3]:
sql_transactions = text(
    "SELECT *\n"
    "FROM public.transactions_raw\n"
    "WHERE status = 'completed'\n"
)

df_transactions = pd.read_sql(sql_transactions, engine)

# Merge transactions with customer attributes (inner join)
df_tx = df_transactions.merge(df_customers, on="customer_id", how="inner")

# Parse transaction_date and derive transaction_month
_tx_dt = pd.to_datetime(df_tx["transaction_date"])
df_tx["transaction_month"] = _tx_dt.dt.to_period("M").dt.to_timestamp()

# Drop the current (incomplete) calendar month to avoid partial-month bias
latest_month = df_tx["transaction_month"].max()
current_period = pd.Timestamp.today().to_period("M").to_timestamp()
if latest_month >= current_period:
    df_tx = df_tx[df_tx["transaction_month"] < current_period].copy()

SEG_ORDER = ["high_value_active", "mid_value_regular", "low_value_dormant", "at_risk_churner"]
SEGMENT_COLORS = {
    "high_value_active": "#4CAF50",
    "mid_value_regular": "#2196F3",
    "low_value_dormant": "#FF9800",
    "at_risk_churner":   "#F44336",
}

print(f"df_customers: {df_customers.shape}")
print(f"df_transactions (completed): {df_transactions.shape}")
print(f"df_tx (merged): {df_tx.shape}")

KeyError: 'transaction_date'

---

## Part 1 — Population Balance <a id="part1"></a>

**Validation goal:** Confirm the generator planted exactly the correct segment proportions (20% / 30% / 30% / 20%) and that each segment's demographic profile matches the design.

[↑ Back to summary](#summary)

### Q1. `true_segment` counts vs planted design

We count customers per `true_segment` and compare against the planted design (20% / 30% / 30% / 20%). This is a **generator sanity check** — `true_segment` is ground truth used only for model validation and is not surfaced on the commercial dashboard.

[↑ Back to summary](#summary)

In [ ]:
# True segment distribution
segment_counts = df_customers["true_segment"].value_counts().sort_values(ascending=False)
segment_pct = segment_counts / segment_counts.sum() * 100

ax = segment_counts.plot(kind="bar", figsize=(10, 5), color="mediumpurple", edgecolor="black")
plt.title("Users by True Segment")
plt.xlabel("True Segment")
plt.ylabel("Number of Users")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)

for i, value in enumerate(segment_counts.values):
    ax.text(i, value + (segment_counts.max() * 0.015), f"{segment_pct.iloc[i]:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

Counts match the planted distribution exactly: **1,600** `high_value_active` (20%), **2,400** `mid_value_regular` (30%), **2,400** `low_value_dormant` (30%), and **1,600** `at_risk_churner` (20%). `true_segment` is intact and can be used as ground truth in cluster recovery and churn validation.

### Q2. Segment profile summary

A single-glance synthesis of the demographic and acquisition profile for each segment — the answer to "Who do we have?" in one table.

[↑ Back to summary](#summary)

In [ ]:
# ── Segment profile summary table ────────────────────────────────────────────

seg_order = ["high_value_active", "mid_value_regular", "low_value_dormant", "at_risk_churner"]

# n and % (from Q1)
n_by_seg   = df_customers["true_segment"].value_counts().reindex(seg_order)
pct_by_seg = (n_by_seg / len(df_customers) * 100).round(1)

# Mean age
mean_age = df_customers.groupby("true_segment")["age"].mean().round(1).reindex(seg_order)

# Top acquisition channel
top_channel = (
    df_customers.groupby(["true_segment", "acquisition_channel"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .groupby("true_segment")
    .first()["acquisition_channel"]
    .reindex(seg_order)
)

# Mean CAC
mean_cac = df_customers.groupby("true_segment")["acquisition_cost"].mean().round(1).reindex(seg_order)

# Top 2 states (validates state is uniform across segments)
top_states_by_seg = (
    df_customers.groupby(["true_segment", "state"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .groupby("true_segment")
    .head(2)
    .groupby("true_segment")["state"]
    .apply(lambda x: ", ".join(x))
    .reindex(seg_order)
)

summary = pd.DataFrame({
    "n":            n_by_seg,
    "% of base":    pct_by_seg,
    "mean_age":     mean_age,
    "top_channel":  top_channel,
    "mean_cac_R$":  mean_cac,
    "top_2_states": top_states_by_seg,
})

display(
    summary.style
    .background_gradient(subset=["mean_age", "mean_cac_R$"], cmap="RdYlGn_r")
    .format({"% of base": "{:.1f}%", "mean_cac_R$": "R${:.1f}", "mean_age": "{:.1f}"})
)

The table confirms the compound narrative: **healthier segments are younger (not always), cheaper to acquire, and skew toward better channels**.

Key observations:
- **`at_risk_churner`** has the highest mean CAC (~R$138) because it over-indexes on `paid_ads`. This creates a compounding disadvantage — the most churn-prone segment is the most expensive to acquire.
- **`high_value_active`** is cheapest to acquire (~R$87), driven by over-indexing on `referral` and `organic`.
- **Top 2 states are SP + MG/RJ for all segments** — confirming that state does not vary meaningfully by segment (no planted signal). This validates the generator design: state is drawn independently of segment.
- **Age spans ~15 years across segments** (from `at_risk_churner` ≈ 27 to `low_value_dormant` ≈ 42) — the most visually striking demographic separator in this dataset.

---

## Part 2 — Demographic Validators <a id="part2"></a>

**Validation goal:** Confirm that the generator correctly planted the demographic correlations — channel bias by segment, age means per segment, segment mix across age bands, and blended CAC by segment.

[↑ Back to summary](#summary)

### Q3. `Acquisition channel` × `true_segment` — does channel bias match design?

Based on the updated logic in `src/fintech_ai_segmentation/faker_base_generation.py`,
the generation defines **channel-to-segment bias** first (`P(segment | channel)`), and then derives
`P(channel | segment)` for customer sampling.

Reference `P(segment | channel)` profiles:

- `organic`: **15%**, **35%**, **30%**, **20%**
- `referral`: **45%**, **35%**, **15%**, **5%**
- `partnership`: **20%**, **40%**, **25%**, **15%**
- `paid_ads`: **10%**, **20%**, **30%**, **40%**

Order above: `high_value_active`, `mid_value_regular`, `low_value_dormant`, `at_risk_churner`.

Derived segment-level channel probabilities (`paid_ads`, `organic`, `referral`, `partnership`):

- `high_value_active`: **11.11%**, **16.67%**, **50.00%**, **22.22%**
- `mid_value_regular`: **15.38%**, **26.92%**, **26.92%**, **30.77%**
- `low_value_dormant`: **30.00%**, **30.00%**, **15.00%**, **25.00%**
- `at_risk_churner`: **50.00%**, **25.00%**, **6.25%**, **18.75%**

This setup makes channel economics and segment composition emerge naturally from the data.

[↑ Back to summary](#summary)

In [ ]:
# Acquisition channel distribution by true_segment (relative %)
segment_order = [
    "high_value_active",
    "mid_value_regular",
    "low_value_dormant",
    "at_risk_churner",
]
channel_order = ["paid_ads", "organic", "referral", "partnership"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.flatten()

for ax, segment in zip(axes, segment_order):
    seg_df = df_customers[df_customers["true_segment"] == segment]
    rel_pct = (
        seg_df["acquisition_channel"]
        .value_counts(normalize=True)
        .reindex(channel_order, fill_value=0)
        .mul(100)
    )

    ax.barh(rel_pct.index, rel_pct.values, color="teal", edgecolor="black")
    ax.set_title(segment)
    ax.set_xlabel("Percentage of users (%)")
    ax.set_ylabel("Acquisition channel")
    ax.set_xlim(0, 60)
    ax.grid(axis="x", alpha=0.3)

    for y, value in enumerate(rel_pct.values):
        ax.text(value + 0.5, y, f"{value:.1f}%", va="center", fontsize=9)

plt.suptitle("Acquisition Channel Mix by True Segment (Relative %)", y=1.02)
plt.tight_layout()
plt.show()

The observed `acquisition_channel` distribution **matches the expected probabilities** from the data-generation process. This pattern is intentional from a business perspective: lower-cost channels (especially `organic` and `referral`) are designed to over-index in healthier segments, while higher-cost `paid_ads` is expected to bring a larger share of `at_risk_churner` customers. In other words, channel mix is not random noise; it encodes realistic go-to-market trade-offs between CAC efficiency and customer quality.

### Q4. Age distribution by segment — do means match generator parameters?

The generator draws `age` from segment-specific Normal distributions (`_AGE_PARAMS_BY_SEGMENT` in `faker_base_generation.py`). We expect:

| Segment | Expected mean age | Expected std |
|---|---|---|
| `high_value_active` | ~38 | ~9 |
| `mid_value_regular` | ~33 | ~10 |
| `low_value_dormant` | ~42 | ~12 |
| `at_risk_churner` | ~27 | ~8 |

[↑ Back to summary](#summary)

In [ ]:
# ── Age vs generator assumptions ─────────────────────────────────────────────

# Generator parameters (from _AGE_PARAMS_BY_SEGMENT in faker_base_generation.py)
age_assumptions = {
    "high_value_active": {"expected_mean": 38, "expected_std": 9},
    "mid_value_regular":  {"expected_mean": 33, "expected_std": 10},
    "low_value_dormant":  {"expected_mean": 42, "expected_std": 12},
    "at_risk_churner":    {"expected_mean": 27, "expected_std": 8},
}

seg_order_age = ["high_value_active", "mid_value_regular", "low_value_dormant", "at_risk_churner"]

# Observed stats
age_observed = (
    df_customers.groupby("true_segment")["age"]
    .agg(observed_mean="mean", observed_std="std", observed_min="min", observed_max="max")
    .round(2)
    .reindex(seg_order_age)
)

age_comparison = age_observed.copy()
age_comparison["expected_mean"] = age_comparison.index.map(lambda s: age_assumptions[s]["expected_mean"])
age_comparison["expected_std"]  = age_comparison.index.map(lambda s: age_assumptions[s]["expected_std"])
age_comparison["mean_delta"]    = (age_comparison["observed_mean"] - age_comparison["expected_mean"]).round(2)
age_comparison["std_delta"]     = (age_comparison["observed_std"]  - age_comparison["expected_std"]).round(2)

display(age_comparison[[
    "expected_mean", "observed_mean", "mean_delta",
    "expected_std",  "observed_std",  "std_delta",
    "observed_min",  "observed_max"
]])

# ── KDE by segment ───────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(11, 5))

for segment in seg_order_age:
    seg_ages = df_customers.loc[df_customers["true_segment"] == segment, "age"]
    color = SEGMENT_COLORS[segment]
    seg_ages.plot.kde(ax=ax, label=segment, color=color, linewidth=2)
    mean_val = seg_ages.mean()
    ax.axvline(mean_val, color=color, linestyle="--", linewidth=1.2, alpha=0.7)

ax.set_title("Age distribution by segment (KDE) — dashed lines = segment means")
ax.set_xlabel("Age")
ax.set_ylabel("Density")
ax.legend(title="Segment", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_xlim(15, 80)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

The observed means match generator parameters within < 1 year for every segment (expected given groups of 1,600–2,400 customers). The standard deviations are marginally tighter than expected — a natural effect of the `age = clip(normal, 18, 80)` bounds applied in the generator.

The KDE reveals the overlap structure:
- **`at_risk_churner`** (mean ≈ 27) and **`low_value_dormant`** (mean ≈ 42) are nearly non-overlapping — age alone cleanly separates these two.
- **`high_value_active`** (mean ≈ 38) and **`low_value_dormant`** (mean ≈ 42) share a substantial tail — **age has limited discriminatory power between them**. Any model using age as a feature should not expect it to cleanly separate these two segments.
- **`mid_value_regular`** (mean ≈ 33) overlaps with both `at_risk_churner` and `high_value_active`.

### Q5. Segment mix within age band — do young customers skew `at_risk_churner`?

Since the generator draws `age` from segment-specific Normal distributions, each age band naturally tilts toward different segments. We expect:
- **18–24** to skew heavily `at_risk_churner` (~45%)
- **55+** to be overwhelmingly `low_value_dormant` (~80%+)
- **35–44** to be closest to the global 20/30/30/20 split

[↑ Back to summary](#summary)

In [ ]:
# ── Segment mix within age band ──────────────────────────────────────────────

df_age = df_customers.copy()
df_age["age_band"] = pd.cut(
    df_age["age"],
    bins=[17, 24, 34, 44, 54, 120],
    labels=["18–24", "25–34", "35–44", "45–54", "55+"],
)

seg_order = ["high_value_active", "mid_value_regular", "low_value_dormant", "at_risk_churner"]
band_order = ["18–24", "25–34", "35–44", "45–54", "55+"]

seg_by_band = (
    df_age.groupby(["age_band", "true_segment"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=band_order, columns=seg_order, fill_value=0)
)

seg_by_band_pct = seg_by_band.div(seg_by_band.sum(axis=1), axis=0) * 100

colors = [SEGMENT_COLORS[s] for s in seg_order]
ax = seg_by_band_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 6),
    color=colors,
    edgecolor="white",
)
ax.set_title("Segment mix by age band (% within band)")
ax.set_xlabel("Age band")
ax.set_ylabel("Share of customers in band (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Segment", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Customer counts — segment × age band:\n")
print(seg_by_band.astype(int).to_string())
print("\nPercentage breakdown:\n")
print(seg_by_band_pct.round(1).to_string())

The stacked bars show the business-facing payoff of the age-segment correlation planted in the generator:

- **18–24:** skews heavily toward **`at_risk_churner`** (~45%) — the youngest band is the highest-churn segment. Campaigns targeting under-25 customers face the highest disengagement risk.
- **25–34:** `at_risk_churner` still elevated above the global average; `mid_value_regular` starts to dominate.
- **35–44:** the most balanced band — closest to the global 20/30/30/20 split. `high_value_active` peaks here.
- **45–54:** `low_value_dormant` begins to dominate.
- **55+:** overwhelmingly **`low_value_dormant`** (~80%+) — older customers are designed to transact rarely and disengage early.

These are **not causal** claims. Age is a **proxy signal**, not a driver — the underlying driver is `true_segment`, which was generated independently of age and then correlated via the sampling design. In production, age would be used as a feature alongside RFM and cohort metrics, not as a standalone predictor.

### Q6. Effective CAC per segment — does blended acquisition cost match design?

The earlier analysis showed **CAC by channel** and **channel bias by segment** separately. Here we connect them: each segment has a different channel mix, so its **blended acquisition cost** differs even if no channel's price changes.

We compute **per-segment CAC statistics** (mean, median, std, p25/p75) directly from `acquisition_cost` grouped by `true_segment`. This gives the **effective (weighted) cost** of acquiring one customer in each segment — the denominator needed for a CAC/LTV ratio.

[↑ Back to summary](#summary)

In [ ]:
# ── Effective (blended) CAC per segment ────────────────────────────────────

seg_order = ["high_value_active", "mid_value_regular", "low_value_dormant", "at_risk_churner"]

cac_seg = (
    df_customers.groupby("true_segment")["acquisition_cost"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
    )
    .round(2)
    .reindex(seg_order)
)

print("Effective CAC by segment:\n")
print(cac_seg.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: boxplot
sns.boxplot(
    data=df_customers,
    x="true_segment",
    y="acquisition_cost",
    order=seg_order,
    palette="Set2",
    ax=axes[0],
)
axes[0].set_title("CAC distribution per segment")
axes[0].set_xlabel("Segment")
axes[0].set_ylabel("Acquisition cost (R$)")
axes[0].tick_params(axis="x", rotation=15)

# Right: mean + IQR bar chart
x = range(len(seg_order))
axes[1].bar(
    x,
    cac_seg["mean"],
    yerr=[cac_seg["mean"] - cac_seg["p25"], cac_seg["p75"] - cac_seg["mean"]],
    color=["#4CAF50", "#2196F3", "#FF9800", "#F44336"],
    capsize=6,
    width=0.5,
)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(seg_order, rotation=15, ha="right")
axes[1].set_title("Mean CAC per segment (bars) + IQR (error bars)")
axes[1].set_xlabel("Segment")
axes[1].set_ylabel("Acquisition cost (R$)")

plt.tight_layout()
plt.show()

**`at_risk_churner`** carries the **highest effective CAC** — driven by its over-index on `paid_ads` (typical ~R$230). **`high_value_active`** has the **lowest blended cost** because it over-indexes on `referral` and `organic`. This creates a compounding disadvantage: the segment most likely to churn is also the most expensive to acquire.

---

## Part 3 — Behavioral Validators <a id="part3"></a>

**Validation goal:** Confirm that the generator planted the correct per-segment activity rates, churn hazards, and seasonal sensitivity coefficients. The behavioral validators use transaction data grouped by `true_segment`.

[↑ Back to summary](#summary)

### Q7. MAU by segment over time — is `high_value_active` stable? Is `at_risk_churner` declining?

**What we're testing:** The generator plants four segments with very different monthly activity rates and churn hazards:

| Segment | Monthly Activity Rate | Monthly Churn Hazard | Expected 12-mo Churn |
|---|---|---|---|
| `high_value_active` | 95% | 1% | ~11% |
| `mid_value_regular` | 85% | 4% | ~38% |
| `low_value_dormant` | 40% | 8% | ~62% |
| `at_risk_churner` | 15% (with decay) | 18% | ~90% |

If the data is generated correctly, we expect `high_value_active` to show a **stable or slowly growing** MAU line (low churn, high per-month activity), while `at_risk_churner` shows a **visible decline** from its early-cohort peak as the 18%/month geometric hazard depletes the population.

[↑ Back to summary](#summary)

In [ ]:
# ── Segment-Level MAU ─────────────────────────────────────────────────────
SEG_ORDER = ["high_value_active", "mid_value_regular", "low_value_dormant", "at_risk_churner"]

mau_by_segment = (
    df_tx
    .groupby(["transaction_month", "true_segment"], as_index=False)["customer_id"]
    .nunique()
    .rename(columns={"customer_id": "mau"})
    .sort_values("transaction_month")
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(
    data=mau_by_segment, x="transaction_month", y="mau",
    hue="true_segment", hue_order=SEG_ORDER, marker="o", ax=ax
)
ax.set_title("Monthly Active Users by segment")
ax.set_xlabel("Month")
ax.set_ylabel("Unique active customers")
ax.legend(title="Segment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Chart read — MAU by segment:**

- **`high_value_active`** should remain **stable or gently rising** — lowest churn hazard (1%/month), highest activity rate (95%). Its trajectory is the best indicator of platform health.
- **`at_risk_churner`** should show a **visible downward slope**, especially in older cohorts (Jan 2022 – mid 2023). The 18%/month geometric hazard means ~50% of any early cohort has permanently exited within 4 months.
- **`low_value_dormant`** typically appears **flat at a low level** — this segment is active in only 40% of months, so most registered customers in this group are silent in any given month even if not yet churned.
- **`mid_value_regular`** sits between the two extremes; expect gradual erosion consistent with 38% 12-month churn.

**Insight vs Caution:** The aggregate MAU plateau is not a platform failure — it is a lagging effect of segment-level churn dynamics. However, if `high_value_active` MAU also flattens or declines, that is a genuine concern that warrants separate monitoring.

### Q8. Seasonality by segment — does `high_value_active` show +20–30% Nov/Dec?

**What we're testing:** The generator applies Brazilian calendar seasonality with **segment-specific sensitivity coefficients**:

| Month(s) | Base Multiplier | Event |
|---|---|---|
| November | +20% | Black Friday + 13th salary (1st instalment) |
| December | +30% | Christmas + 13th salary (2nd instalment) |
| January | −20% | Post-holiday hangover |
| February | −15% | Carnival lull |

Each segment responds with different intensity: `effective_lift = (base_lift) × sensitivity`

| Segment | Sensitivity | Nov effective lift | Dec effective lift |
|---|---|---|---|
| `high_value_active` | **1.00** | +20% | +30% |
| `mid_value_regular` | 0.70 | +14% | +21% |
| `low_value_dormant` | 0.30 | +6% | +9% |
| `at_risk_churner` | **0.20** | +4% | +6% |

We normalise TPV by **unique active customers** per month to remove the population-size effect and surface the **per-customer spend signal** directly.

[↑ Back to summary](#summary)

In [ ]:
# ── Seasonality decomposition — TPV per active customer ───────────────────

# Monthly TPV per segment
tpv_by_seg = (
    df_tx
    .groupby(["transaction_month", "true_segment"], as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "total_amount"})
)

# Merge with segment MAU for normalisation
tpv_norm = tpv_by_seg.merge(mau_by_segment, on=["transaction_month", "true_segment"])
tpv_norm["tpv_per_active"] = tpv_norm["total_amount"] / tpv_norm["mau"]
tpv_norm = tpv_norm.sort_values("transaction_month")

fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(
    data=tpv_norm, x="transaction_month", y="tpv_per_active",
    hue="true_segment", hue_order=SEG_ORDER, marker="o", ax=ax
)
# Annotate Nov/Dec peaks
for mo_str, label in [("2023-11-01", "Nov '23"), ("2023-12-01", "Dec '23"),
                       ("2024-11-01", "Nov '24"), ("2024-12-01", "Dec '24")]:
    ax.axvline(pd.Timestamp(mo_str), color="grey", linestyle="--", alpha=0.4)

ax.set_title("Monthly TPV per active customer by segment (R$)")
ax.set_xlabel("Month")
ax.set_ylabel("R$ per active customer")
ax.legend(title="Segment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Nov–Dec 2024 amplitude check
print("=== Nov–Dec 2024 seasonal lift vs Sep–Oct 2024 baseline ===")
for seg in SEG_ORDER:
    sub = tpv_norm[tpv_norm["true_segment"] == seg].copy()
    sub["ym"] = sub["transaction_month"].dt.to_period("M").astype(str)
    base = sub[sub["ym"].isin(["2024-09", "2024-10"])]["tpv_per_active"].mean()
    peak = sub[sub["ym"].isin(["2024-11", "2024-12"])]["tpv_per_active"].mean()
    lift = (peak / base - 1) * 100 if base > 0 else float("nan")
    print(f"  {seg:<25}: baseline R${base:>8,.0f}  peak R${peak:>8,.0f}  lift {lift:>+.1f}%")

**Chart read — TPV per active customer by segment:**

If the generator's seasonal logic ran correctly, **`high_value_active` should show the sharpest Nov/Dec spikes** (~+20–30% above its own baseline), while **`at_risk_churner` should be nearly flat** (~+4–6%). The ratio of observed amplitudes across segments validates the sensitivity coefficients (1.0 : 0.70 : 0.30 : 0.20).

The normalisation by active customers is critical: without it, `high_value_active` dominates simply because it has more active users — the per-customer view isolates the **spending behaviour** of each segment.

**Business implication:** November and December campaigns should be **disproportionately targeted** at `high_value_active` and `mid_value_regular` — they capture most of the incremental spend. Running the same promotions to `at_risk_churner` customers delivers ~5× less lift per R$ of campaign cost.

---

## Part 4 — Retention Validators <a id="part4"></a>

**Validation goal:** Confirm that the planted churn hazards produce the expected M0–M6 retention curve separation, and that channel quality rankings are confounded by segment composition (not intrinsic channel effects).

[↑ Back to summary](#summary)

### Setup — build cohort pivot for retention analysis

The retention curves require a cohort-based tenure pivot table. We reuse the same window and methodology as Notebook 2 (M6-eligible cohorts only).

[↑ Back to summary](#summary)

In [ ]:
# ── Cohort pivot setup ────────────────────────────────────────────────────────

window_start = pd.Timestamp("2023-01-01")
window_end   = pd.Timestamp("2026-02-28")

RETENTION_HORIZONS = [3, 6]
MAX_TENURE = max(RETENTION_HORIZONS)

df_tx_window = df_tx[df_tx["transaction_month"].between(window_start, window_end)].copy()

# Monthly activity flags per customer
activity = (
    df_tx_window.groupby(["customer_id", "transaction_month"])
    .size()
    .reset_index(name="tx_count")
    .rename(columns={"transaction_month": "calendar_month"})
)
activity["active"] = 1

# Cohort month = registration month
df_customers["cohort_month"] = pd.to_datetime(df_customers["registration_date"]).dt.to_period("M").dt.to_timestamp()
customers = df_customers[["customer_id", "acquisition_channel", "cohort_month"]].copy()

customers_window = customers[
    customers["cohort_month"].between(window_start, window_end)
].copy()

latest_complete_month = df_tx_window["transaction_month"].max().to_period("M")

# Build tenure grid
tenure_df = pd.DataFrame({"tenure_index": list(range(MAX_TENURE + 1))})
cust_for_grid = customers_window.copy()
cust_for_grid["key"] = 1
tenure_df["key"] = 1
grid = cust_for_grid.merge(tenure_df, on="key").drop(columns="key")
grid["calendar_month"] = grid.apply(
    lambda r: r["cohort_month"] + pd.DateOffset(months=r["tenure_index"]),
    axis=1
)
grid["calendar_month"] = pd.to_datetime(grid["calendar_month"]).dt.to_period("M").dt.to_timestamp()

grid = grid.merge(activity[["customer_id", "calendar_month", "active"]], on=["customer_id", "calendar_month"], how="left")
grid["active"] = grid["active"].fillna(0).astype(int)

pivot = grid.pivot_table(
    index="customer_id",
    columns="tenure_index",
    values="active",
    aggfunc="max",
    fill_value=0,
)
rename_map = {i: f"m{i}" for i in range(MAX_TENURE + 1)}
pivot = pivot.rename(columns=rename_map).reset_index()
pivot = pivot.merge(customers_window[["customer_id", "acquisition_channel", "cohort_month"]], on="customer_id")

for i in range(MAX_TENURE + 1):
    if f"m{i}" not in pivot.columns:
        pivot[f"m{i}"] = 0

# Eligibility flags: customer's cohort month + horizon must be <= latest complete month
for h in RETENTION_HORIZONS:
    pivot[f"eligible_h{h}"] = (
        pivot["cohort_month"] + pd.DateOffset(months=h)
    ).dt.to_period("M") <= latest_complete_month

print(f"Pivot shape: {pivot.shape}")
print(f"M6-eligible cohorts: {pivot['eligible_h6'].sum():,}")

### Q9. Retention curves by segment (M0–M6) — expected separation: `high_value` > `at_risk`

The aggregate retention curve hides **very different lifecycle shapes per segment**. Using the generator's planted churn hazards, we can derive the **expected M6 survival** (probability of not yet having permanently churned):

| Segment | Monthly Churn Hazard | Expected M6 survival `(1−h)^6` |
|---|---|---|
| `high_value_active` | 1% | ~94% |
| `mid_value_regular` | 4% | ~78% |
| `low_value_dormant` | 8% | ~61% |
| `at_risk_churner` | 18% | ~35% |

> **Note:** the **observed M6 active rate** will be lower than the survival floor because surviving customers can still be inactive in a given month (governed by `p_active_per_month`). For `at_risk_churner` (15% monthly activity), even surviving customers are usually silent — so observe rates well below the ~35% survival estimate.

If the segments are separable, the four retention curves should **clearly fan out** — this is the most direct in-notebook signal that K-Means clustering in Notebook 3 has clean behavioural signal to work with.

[↑ Back to summary](#summary)

In [ ]:
# ── Segment-level tenure retention curves ─────────────────────────────────────

# Add true_segment to pivot (pivot was built from customers_window which
# selected only customer_id and acquisition_channel — merge it back in)
cust_seg = df_customers[["customer_id", "true_segment"]]
pivot_seg = pivot.merge(cust_seg, on="customer_id", how="left")
pivot_seg_m6 = pivot_seg[pivot_seg["eligible_h6"]].copy()

# Retention rate per tenure slot per segment
seg_ret_rows = []
for seg, g in pivot_seg_m6.groupby("true_segment"):
    n = len(g)
    for slot in range(MAX_TENURE + 1):
        col = f"m{slot}"
        if col in g.columns:
            seg_ret_rows.append({
                "true_segment": seg,
                "tenure_slot": slot,
                "active_rate": g[col].sum() / n,
                "n": n,
            })

seg_ret_df = pd.DataFrame(seg_ret_rows).sort_values(["true_segment", "tenure_slot"])

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(
    data=seg_ret_df, x="tenure_slot", y="active_rate",
    hue="true_segment", hue_order=SEG_ORDER, marker="o", ax=ax
)
# Expected M6 survival reference lines
M6_EXPECTED = {
    "high_value_active": 0.94,
    "mid_value_regular": 0.78,
    "low_value_dormant": 0.61,
    "at_risk_churner":   0.35,
}
for seg, val in M6_EXPECTED.items():
    ax.axhline(val, linestyle=":", alpha=0.4, color="grey")
    ax.text(MAX_TENURE + 0.1, val, f"{val:.0%} (design)", va="center", fontsize=8, color="grey")

ax.set_title("Tenure retention curves by segment (M6-eligible cohorts)")
ax.set_xlabel("Tenure month (M0 = registration month)")
ax.set_ylabel("Share active (≥1 completed tx)")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xticks(range(MAX_TENURE + 1))
ax.set_xticklabels([f"M{i}" for i in range(MAX_TENURE + 1)])
ax.legend(title="Segment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Observed vs expected table
print("=== M6 observed active rate vs generator design ===")
print(f"{'Segment':<25} {'Obs M6 rate':>12} {'Design survival':>16} {'Gap':>8}")
for seg in SEG_ORDER:
    sub = seg_ret_df[(seg_ret_df["true_segment"] == seg) & (seg_ret_df["tenure_slot"] == MAX_TENURE)]
    if len(sub) > 0:
        obs = sub["active_rate"].iloc[0]
        exp = M6_EXPECTED.get(seg, float("nan"))
        print(f"{seg:<25} {obs:>11.1%} {exp:>15.1%} {obs - exp:>+7.1%}")

**Chart read — segment retention curves:**

The four curves should **clearly separate** with `high_value_active` at the top and `at_risk_churner` near the bottom. The dotted grey lines mark the **expected M6 survival** from the generator design.

**Validation check:** If the rank ordering holds (high_value > mid_value > low_value > at_risk) and the spread is wide (not all lines compressed near 50%), the generator has successfully planted distinguishable behavioural segments. The gap table below the chart shows how much of the difference between observed and expected is attributable to the `p_active_per_month` intermittency effect on top of churn.

**Why this matters for Notebook 3:** K-Means clustering performs best when the underlying populations are already separated in feature space. These retention curves are the first visual confirmation that the segments produce different transaction histories — the essential precondition for a meaningful clustering result.

### Q10. Channel × segment composition heatmap — confound audit

The rolling channel retention charts in Notebook 2 show **Referral** consistently leading on M6 and strict-streak. Before attributing this to intrinsic "channel quality," we must ask: **is Referral's strength mainly explained by the type of customer it attracts?**

The generator explicitly biases acquisition channels toward certain segments:

| Channel | `high_value_active` | `mid_value_regular` | `low_value_dormant` | `at_risk_churner` |
|---|---|---|---|---|
| **Referral** | ~40% | ~30% | ~20% | ~10% |
| **Organic** | ~20% | ~30% | ~30% | ~20% |
| **Partnership** | ~15% | ~30% | ~30% | ~25% |
| **Paid_ads** | ~10% | ~25% | ~30% | ~35% |

If the data is generated correctly, the heatmap below should confirm Referral with the highest `high_value_active` share and Paid_ads with the highest `at_risk_churner` share.

[↑ Back to summary](#summary)

In [ ]:
# ── Channel × Segment composition ────────────────────────────────────────────

ch_seg_pct = (
    df_customers
    .groupby(["acquisition_channel", "true_segment"])
    .size()
    .reset_index(name="n")
)
ch_seg_pct["pct"] = ch_seg_pct.groupby("acquisition_channel")["n"].transform(
    lambda x: x / x.sum() * 100
)

heatmap_df = ch_seg_pct.pivot(
    index="acquisition_channel", columns="true_segment", values="pct"
)[SEG_ORDER]

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(heatmap_df.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=50)
ax.set_xticks(range(len(SEG_ORDER)))
ax.set_xticklabels([s.replace("_", "\n") for s in SEG_ORDER], fontsize=9)
ax.set_yticks(range(len(heatmap_df)))
ax.set_yticklabels(heatmap_df.index, fontsize=10)
ax.set_title("Segment composition by acquisition channel (% of channel's customers)", pad=10)
plt.colorbar(im, ax=ax, label="%")
for i in range(len(heatmap_df)):
    for j in range(len(SEG_ORDER)):
        val = heatmap_df.values[i, j]
        ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                color="black" if val < 35 else "white", fontsize=9)
plt.tight_layout()
plt.show()

print("\n=== Channel × Segment composition (%) ===")
display(heatmap_df.round(1))

**Chart read — channel × segment composition:**

This heatmap is the **confound audit** for the retention channel rankings. If Referral's row is disproportionately dark in the `high_value_active` column and Paid_ads is dark in `at_risk_churner`, the retention gap between channels reflects **who arrives**, not just how well each channel converts or retains customers post-acquisition.

**Key implication:** Do not scale Referral campaigns based on M6 rates alone — understand whether the segment mix is driven by the incentive design, referrer profile, or product-market fit. If a new referral tier attracts at-risk customers (e.g., discount-seekers), Referral's M6 rate will degrade toward Paid_ads levels.

**Insight vs Caution:** Channel and segment are confounded by design in this synthetic dataset. The same decomposition is essential in real-world analysis before setting channel-level CAC budgets or M6 thresholds.

---

## Part 5 — Generator Audit Summary <a id="part5"></a>

**Validation goal:** A single-table reconciliation of observed vs. expected values for every key planted parameter. Any row marked ⚠ requires investigation before using the data for model training.

Tolerances used:
- **Count tolerance:** ±50 customers (< 1% of smallest segment)
- **Age mean tolerance:** ±2 years

[↑ Back to summary](#summary)

In [ ]:
# ── Generator Audit Summary Table ────────────────────────────────────────────

TOTAL_CUSTOMERS = len(df_customers)

# Expected values (planted by generator)
expected_counts = {
    "high_value_active": 1600,
    "mid_value_regular": 2400,
    "low_value_dormant": 2400,
    "at_risk_churner":   1600,
}
expected_count_pct = {
    "high_value_active": 20.0,
    "mid_value_regular": 30.0,
    "low_value_dormant": 30.0,
    "at_risk_churner":   20.0,
}
expected_age_means = {
    "high_value_active": 38,
    "mid_value_regular": 33,
    "low_value_dormant": 42,
    "at_risk_churner":   27,
}

COUNT_TOLERANCE = 50
AGE_TOLERANCE   = 2

# Observed values
obs_counts   = df_customers["true_segment"].value_counts()
obs_age_mean = df_customers.groupby("true_segment")["age"].mean()

audit_rows = []

# --- Population balance rows ---
for seg in SEG_ORDER:
    exp_n  = expected_counts[seg]
    obs_n  = obs_counts.get(seg, 0)
    exp_pct = expected_count_pct[seg]
    obs_pct = round(obs_n / TOTAL_CUSTOMERS * 100, 1)
    status  = "✓" if abs(obs_n - exp_n) <= COUNT_TOLERANCE else "⚠"
    audit_rows.append({
        "Parameter": f"{seg} count",
        "Expected":  f"{exp_n:,} ({exp_pct:.0f}%)",
        "Observed":  f"{obs_n:,} ({obs_pct:.1f}%)",
        "Delta":     obs_n - exp_n,
        "Tolerance": f"±{COUNT_TOLERANCE}",
        "Status":    status,
    })

# --- Age mean rows ---
for seg in SEG_ORDER:
    exp_age = expected_age_means[seg]
    obs_age = round(obs_age_mean.get(seg, float("nan")), 2)
    delta   = round(obs_age - exp_age, 2)
    status  = "✓" if abs(delta) <= AGE_TOLERANCE else "⚠"
    audit_rows.append({
        "Parameter": f"mean age — {seg}",
        "Expected":  f"~{exp_age}",
        "Observed":  f"{obs_age:.2f}",
        "Delta":     delta,
        "Tolerance": f"±{AGE_TOLERANCE}",
        "Status":    status,
    })

audit_df = pd.DataFrame(audit_rows)

def highlight_status(row):
    color = "background-color: #c8e6c9" if row["Status"] == "✓" else "background-color: #ffcdd2"
    return [""] * (len(row) - 1) + [color]

display(
    audit_df.style
    .apply(highlight_status, axis=1)
    .set_caption("Generator Audit — Observed vs Expected (green = within tolerance, red = outside tolerance)")
    .hide(axis="index")
)

n_pass = (audit_df["Status"] == "✓").sum()
n_fail = (audit_df["Status"] == "⚠").sum()
print(f"\nAudit result: {n_pass} checks passed ✓ | {n_fail} checks flagged ⚠")
if n_fail > 0:
    print("\nFlagged parameters:")
    print(audit_df[audit_df["Status"] == "⚠"][["Parameter", "Expected", "Observed", "Delta"]].to_string(index=False))

**Audit interpretation:**

All ✓ rows confirm that the generator produced data within the expected tolerance bounds. Any ⚠ row indicates a parameter that drifted outside tolerance — typically caused by:
1. A code change in `faker_base_generation.py` that altered segment proportions or age distributions
2. A random seed change that produced an unusually skewed sample
3. A post-generation data manipulation that introduced bias

**Action for ⚠ rows:** Regenerate the synthetic data via `generate_all_tables()` and re-run this notebook. If the issue persists, investigate the relevant section of `faker_base_generation.py`.

---

## Notebook summary — what we covered

This notebook implements **generator validation** using `true_segment` ground-truth labels that are hidden from Notebooks 1 and 2. The four validation parts cover:

1. **Population balance** — segment counts match 20/30/30/20 planted design
2. **Demographic validators** — channel bias, age means, age-band segment mix, and blended CAC all match generator parameters
3. **Behavioral validators** — MAU trajectories and seasonality amplitudes match planted activity rates and sensitivity coefficients
4. **Retention validators** — M0–M6 curves separate by segment as expected from planted churn hazards; channel × segment composition confirms retention differences are confounded by segment mix
5. **Generator audit summary** — single-table pass/fail reconciliation for all key planted parameters

If all checks pass, the synthetic dataset is ready to be treated as a realistic training and evaluation environment for the segmentation and churn models in downstream notebooks.